# Certificates, and what restarts are worth

This notebook accompanies the docs page
[`global-certification`](../../docs/examples/global-certification.md). Exchange stability is a
statement about a neighborhood, and the neighborhood of single-row relocations is small. On a
small problem the global optimum can be *proved*, which turns "did the fit find the best
labeling" into a measurable event — and lets the value of extra restarts be priced.

Set `SCOREQUANT_EXAMPLE_FAST=1` to shrink the restart sweep and the scaling sweep for a quick
pass.

## Double precision is an application choice

The library never sets global numerical configuration at import time, so the notebook turns
double precision on itself, before anything computes.

In [ ]:
import jax

jax.config.update("jax_enable_x64", True)

## The problem, and its certificate

Two overlapping spectral templates, twenty-eight events, exact linear component scores, and
the reference intensity as each event's weight. Twenty-eight is deliberately tiny: global
certification is exponential in the number of distinct positive-weight score atoms, and
refuses larger instances by name rather than appearing to hang.

In [ ]:
import scorequant as sq
from examples.global_certification import certification_table, make_figure, run_study

split = certification_table(28)
certificate = sq.certify_partition(split.scores, weights=split.weights, n_bins=5)

print("events                ", split.scores.shape[0])
print("score columns         ", split.scores.shape[1])
print("certificate status    ", certificate.status)
print("certified objective   ", round(certificate.objective, 6))
print("outstanding gap       ", certificate.gap)
print("search nodes explored ", certificate.nodes_explored)

## Two incumbents, two outcomes

The search starts from an incumbent so that pruning is effective from the first node, so
handing it the labels of an exchange result answers the practical question directly. Both
answers occur, and both are reported through the same fields.

In [ ]:
from examples.global_certification import incumbent_cases

print(f"{'case':<32}{'rows':>6}{'incumbent':>12}{'certified':>12}{'gain':>11}{'nodes':>8}")
print("-" * 81)
for case in incumbent_cases():
    print(
        f"{case.label:<32}{case.n_rows:>6}{case.incumbent_objective:>12.6f}"
        f"{case.certified_objective:>12.6f}{case.gain:>11.6f}{case.nodes_explored:>8}"
    )

The second incumbent is exchange-stable — no single relocation improves it — and the search
still finds a labeling worth 0.047 nat more. Local optimality of any kind is a statement about
a neighborhood.

## Restarts, measured against the certificate

`run_study` runs the whole thing: both incumbent cases, a restart sweep over two seeding modes
on the certified problem, a scaling sweep of the certification itself, and one run that
deliberately exhausts its node budget. Every trial is a real fit with `n_restarts` set, not a
maximum reconstructed afterwards.

In [ ]:
study = run_study()
rates = study.metrics["hit_rates"]
rows = rates["rows"]
modes = list(dict.fromkeys(row["init"] for row in rows))
restarts = list(dict.fromkeys(row["n_restarts"] for row in rows))
table = {(row["init"], row["n_restarts"]): row for row in rows}

header = f"{'restarts':>9}" + "".join(f"{mode:>16}" for mode in modes) + f"{'seconds per fit':>18}"
print(header)
print("-" * len(header))
for n_restarts in restarts:
    line = f"{n_restarts:>9}"
    for mode in modes:
        line += f"{table[(mode, n_restarts)]['hit_rate']:>16.3f}"
    line += f"{table[(modes[0], n_restarts)]['seconds_per_trial']:>18.3f}"
    print(line)
print()
print("proving the optimum cost", round(rates["certified_seconds"], 2), "seconds")

Three separate claims come out of that table.

One restart is not an answer: a single seeded exchange reaches the global optimum in about a
third of trials here, and nothing about the run announces which third it is in. Restarts buy
most of the gap cheaply, reaching about 95% at six restarts for a small fraction of what
proving the optimum costs, and then flatten. And seeding is not a detail: random
initialization finds the optimum far less often *and* costs more per restart, because a random
labeling starts far from any sensible geometry and the exchange has to relocate its way out.

## What certification costs

In [ ]:
scaling = study.metrics["scaling"]
budgets = list(dict.fromkeys(row["n_bins"] for row in scaling))
sizes = list(dict.fromkeys(row["n_rows"] for row in scaling))
lookup = {(row["n_bins"], row["n_rows"]): row for row in scaling}

header = f"{'atoms':>7}" + "".join(f"{f'{b} cells':>14}" for b in budgets)
print(header)
print("-" * len(header))
for n_rows in sizes:
    line = f"{n_rows:>7}"
    for n_bins in budgets:
        line += f"{lookup[(n_bins, n_rows)]['nodes_explored']:>14,}"
    print(line)
print()
overrun = study.metrics["overrun"]
print(
    f"{overrun['n_rows']} atoms at {overrun['n_bins']} cells: {overrun['status']} after "
    f"{overrun['nodes_explored']:,} nodes, {overrun['gap']:.4f} nat still outstanding"
)

Read the columns down. The tree grows by roughly one and a half times per additional atom, and
faster at larger cell budgets. That is what "exponential" means in practice, and it is why the
capacity guard exists rather than a progress bar. The counts are not perfectly monotone in the
atom count, because a stronger incumbent prunes more of the tree; the trend is what matters.

## The committed figure

In [ ]:
figure = make_figure(study)
figure

## Interpretation

Certification does not scale to a real sample and was never meant to. What it does is
calibrate the thing that does scale: certify a small instance of the problem you care about,
measure how many restarts it takes to reach the certified optimum, and spend that many on the
full sample, where the same measurement is impossible and the same optimizer is running.

Two properties of the certificate make that workflow safe. It always reports which of two
things happened — an exhausted tree, or an exhausted budget with a genuine outstanding ceiling
— and it refuses instances beyond its declared capacity by name. It is also D-only by
construction: the singleton-completion bound rests on Loewner monotonicity of the log
determinant under refinement, which the profiled Schur objective does not inherit. The theory
is [Chapter 8](../../docs/book/ch08-d-optimality.md).